# 3 · Audio segmentation & forced alignment (MMS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/Koelsch-Phoneme-Recognition/blob/main/03_segmentation/03_mms_segmentation.ipynb)

**Pipeline stage 3 of 6.** The recordings are 2–5-minute narratives with no
internal time alignment, whereas CTC training needs short, aligned utterances.
We use **Meta's MMS forced aligner** to get word-level timestamps, then compare
three segmentation strategies:

1. **10-word** fixed windows — long, context-rich, but cut across pauses.
2. **5-word** fixed windows — short, uniform, tighter alignment.
3. **Prosodic (breath-group)** — cut at boundary signals in the orthography;
   **adopted** because boundaries coincide with real acoustic pauses.

## Setup

In [ ]:
!pip -q install torch torchaudio transformers soundfile librosa
import torch, torchaudio, soundfile as sf, os, glob, json, re
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "·", device)

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

# On Google Colab: clone the repo once (or mount Drive and point _root at it).
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass

# Repo root = the folder that contains kolsch_paths.py (found from any subfolder).
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)                       # any remaining relative paths resolve at the root
print("repo root:", ROOT)

In [ ]:
import glob, pandas as pd
index = pd.read_csv(INDEX)

def read_body(path, drop_header=2):
    lines=[l for l in open(path,encoding="utf-8").read().splitlines() if l.strip()]
    return " ".join(lines[drop_header:])
print(len(index),"recording(s):", list(index["id"]))

## 1 · Standardise audio (mono, 16 kHz)

In [ ]:
TARGET_SR = 16000
def load_audio(path):
    wav, sr = sf.read(path)
    if wav.ndim > 1: wav = wav.mean(axis=1)         # to mono
    wav = torch.tensor(wav, dtype=torch.float32)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    wav = wav / (wav.abs().max() + 1e-9)            # peak normalise
    return wav

## 2 · Forced alignment with MMS

We use torchaudio's MMS forced-alignment bundle (`MMS_FA`, pre-trained on
1,100+ languages). It returns frame-level token alignments that we aggregate
into **word spans**. Romanisation handles the Latin-based Kölsch orthography.

In [ ]:
from torchaudio.pipelines import MMS_FA as bundle
import unicodedata, re

aligner_model = bundle.get_model().to(device)
tokenizer = bundle.get_tokenizer()      # romanised words -> token ids
aligner   = bundle.get_aligner()        # runs forced_align + merges into word spans

_FOLD = {"ä":"a","ö":"o","ü":"u","ß":"ss","ï":"i","ë":"e","é":"e","è":"e",
         "á":"a","à":"a","â":"a","ê":"e","î":"i","ô":"o","û":"u","ç":"c"}
def romanise_word(w):
    """Fold Kölsch spelling to the aligner's a-z alphabet."""
    w = w.lower()
    for k,v in _FOLD.items(): w = w.replace(k, v)
    w = "".join(c for c in unicodedata.normalize("NFD", w)
                if unicodedata.category(c) != "Mn")     # strip leftover accents
    return re.sub(r"[^a-z]", "", w)                     # drop -, ', digits, punctuation

def align_words(wav, transcript):
    """Return [(original_word, start_sec, end_sec), ...] via MMS forced alignment.

    Aligns on ROMANISED a-z tokens, so hyphens/apostrophes no longer collide with
    the CTC blank index (0) — that was the `targets shouldn't contain blank index`
    error. Words that reduce to empty (stray '-', digits) are dropped. The ORIGINAL
    word (with punctuation) is kept so prosodic cues survive downstream."""
    pairs = [(w, romanise_word(w)) for w in transcript.split()]
    pairs = [(w, r) for w, r in pairs if r]             # drop empties -> fixes the ValueError
    words = [r for _, r in pairs]
    with torch.inference_mode():
        emission, _ = aligner_model(wav.unsqueeze(0).to(device))
    token_spans = aligner(emission[0], tokenizer(words))
    ratio = wav.size(0) / emission.size(1) / TARGET_SR
    return [(w, sp[0].start * ratio, sp[-1].end * ratio)
            for (w, _), sp in zip(pairs, token_spans)]
# If your torchaudio lacks get_aligner()/get_tokenizer(), pip install a newer
# torchaudio, or use the `ctc-forced-aligner` package (see README).

## 3 · Three segmentation strategies

### (a) Fixed windows — 5-word and 10-word

In [ ]:
def fixed_windows(word_spans, n=5):
    chunks = []
    for i in range(0, len(word_spans), n):
        grp = word_spans[i:i+n]
        if grp:
            chunks.append((grp[0][1], grp[-1][2], " ".join(w for w,_,_ in grp)))
    return chunks   # [(start, end, text), ...]

### (b) Prosodic (breath-group) segmentation — adopted

Cut at boundary signals already present in the orthography:
- **Hard** boundaries (full breath reset) at sentence-final punctuation `. ! ? ;`
- **Soft** boundaries (clause pause) at a comma before a Kölsch clause-starter
  (`un, dann, da, do, wie, ävver, odder, weil, dat, wenn, als, so, su`) or after
  a closing discourse particle (`ne, jo, ja, also`).

Then merge chunks < 3 words and split chunks > 12 words (target band 3–10).

In [ ]:
CLAUSE_STARTERS = {"un","dann","da","do","wie","ävver","odder","weil","dat",
                   "wenn","als","so","su"}
CLOSERS = {"ne","jo","ja","also"}

def prosodic_chunks(word_spans):
    """Breath-group segmentation. Reads boundary cues from the word text carried
    in word_spans (each entry is (original_word, start, end))."""
    B = set()
    for i,(w,_,_) in enumerate(word_spans):
        clean = re.sub(r"[^\wäöüßçəɪɛɔʊʁ']", "", w.lower())
        nxt = re.sub(r"\W","",word_spans[i+1][0].lower()) if i+1 < len(word_spans) else ""
        if re.search(r"[.!?;]$", w):                        # hard boundary
            B.add(i)
        elif w.endswith(",") and nxt in CLAUSE_STARTERS:    # soft boundary
            B.add(i)
        elif clean in CLOSERS:
            B.add(i)
    chunks, cur = [], []
    for i, ws in enumerate(word_spans):
        cur.append(ws)
        if i in B:
            chunks.append(cur); cur = []
    if cur: chunks.append(cur)
    out = []                                                # merge <3, split >12
    for ch in chunks:
        if out and len(ch) < 3:
            out[-1].extend(ch)
        elif len(ch) > 12:
            for j in range(0, len(ch), 10): out.append(ch[j:j+10])
        else:
            out.append(ch)
    return [(c[0][1], c[-1][2], " ".join(w for w,_,_ in c)) for c in out if c]

## 4 · Export segments + manifest

Each chunk is written as a 16-kHz WAV plus a manifest row
(`audio_path, text, start, end`) for the training notebook.

In [ ]:
import json
def export_segments(wav, chunks, out_dir, stem):
    os.makedirs(out_dir, exist_ok=True)
    rows=[]
    for k,(s,e,text) in enumerate(chunks):
        clip = wav[int(s*TARGET_SR):int(e*TARGET_SR)]
        p = os.path.join(out_dir, f"{stem}_{k:03d}.wav")
        sf.write(p, clip.numpy(), TARGET_SR)
        rows.append({"id": stem, "audio_path": p, "text": text, "start": s, "end": e})
    return rows

# GLOBAL: forced-align + segment every recording in the registry -> manifest.csv
# (needs the MMS aligner weights; downloads on first run.)
manifest=[]
for r in index.itertuples(index=False):
    wav   = load_audio(os.path.join(DATA, r.audio))
    body  = read_body(os.path.join(DATA, r.transcript))
    spans = align_words(wav, body)                 # MMS forced alignment
    chunks = prosodic_chunks(spans)                # adopted breath-group strategy
    manifest += export_segments(wav, chunks, SEG, r.id)
    print(f"{r.id}: {len(chunks)} segments from {len(body.split())} words")

if manifest:
    pd.DataFrame(manifest).to_csv(os.path.join(SEG,"manifest.csv"), index=False)
    print("wrote", os.path.join(SEG,"manifest.csv"), "->", len(manifest), "utterances")

## Why prosodic wins

Fixed windows are deterministic but ignore where the speaker breathes. Prosodic
segments place boundaries at real acoustic pauses, giving cleaner alignment
targets and lower downstream error. Discard any chunk that fails a
duration/speech-rate sanity check before training.